In [1]:
# Standard library
import json
import random
import time
from argparse import ArgumentParser

# Third-party
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.profilers import AdvancedProfiler

# First-party
from neural_lam import constants, utils, config
from neural_lam.weather_dataset import WeatherDataset
from neural_lam.downscaling_dataset import DownscalingDataset
from neural_lam.netCDF_dataset import NetCDFDataset
from neural_lam.models.graph_efm import GraphEFM
from neural_lam.models.graph_fm import GraphFM
from neural_lam.models.graphcast import GraphCast
from neural_lam.models.diffusion import Diffusion
from neural_lam.models.ir_sde import IR_SDE
from neural_lam.models.stochastic_interpolants import SI

In [2]:
MODELS = {
    "graphcast": GraphCast,
    "graph_fm": GraphFM,
    "graph_efm": GraphEFM,
    "diffusion": Diffusion,
    "ir_sde": IR_SDE, 
    "SI": SI,
}

In [3]:
config_loader = config.Config.from_file('neural_lam/clim_config.yaml')

In [4]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=1,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.clt_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.hus_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.pr_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_mm_day_noleap.nc', 'standardized.psl_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.tas_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ta_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.ua_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc', 'standardized.va_day_EC-Earth3-Veg_historical_r1i1p1f1_gr_1951-2014_noleap.nc']
Reading static fields from ['standardized.remapped.orog_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_fx.nc', 'scaled.remapped.sftlf_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_fx.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [5]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [6]:
test_batch['LQ'].shape

torch.Size([1, 30, 400, 550])

In [7]:
test_batch['HQ'].shape

torch.Size([1, 2, 400, 550])